In [1]:
"""
build_toll_silver.py

Creates a Toll "silver" table for I-66 Inside the Beltway toll trip pricing data.

Key logic (per your notes):
1) Use interval-based assignment (as-of join). Never round or bin toll timestamps.
2) Each toll record is valid from its timestamp until the next toll update.
3) For each 5-min probe timestamp, assign the most recent toll where valid_from <= ts_5min.
4) Use the "headline toll" by selecting the longest OD toll each time step:
   - EB: StartZoneID 3100 -> EndZoneID 3130
   - WB: StartZoneID 3200 -> EndZoneID 3230
5) Treat toll as corridor-level signal. Assign same toll value to all TMCs downstream.
6) Create dynamic features: toll_change_5min, rolling mean, rolling std.

Input example columns expected (CSV):
CalculatedDateTime, IntervalDateTime, IntervalEndDateTime, CorridorID, CorridorName,
StartZoneID, StartZoneName, EndZoneID, EndZoneName, ZoneTollRate, source_file

Output (Parquet):
ts_5min, direction, corridor_id, corridor_name, start_zone_id, end_zone_id,
toll_corridor_usd, toll_change_5min, toll_rollmean_15min, toll_rollstd_30min, source_tag

Author: ChatGPT
"""

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Optional, Tuple, List

import numpy as np
import pandas as pd
import pyarrow


# ----------------------------
# User configuration
# ----------------------------
BASE_DIR = Path(r"C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data")

TOLL_MONTHLY_DIR = BASE_DIR / "a_raw_download" / "i66_toll_trip_pricing_monthly" / "csv_monthly" / "2023"

OUT_DIR = BASE_DIR / "b_processed_data" / "tolls" / "i66_itb"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Headline OD pairs
HEADLINE_PAIRS = {
    "EB": (3100, 3130),
    "WB": (3200, 3230),
}

# Probe grid frequency
PROBE_FREQ = "5min"
LOCAL_TZ = "America/New_York"
TOLL_START_LOCAL_HHMM = "05:30"


In [2]:
# ----------------------------
# Helpers
# ----------------------------
def _infer_direction(corridor_name: str) -> Optional[str]:
    if not isinstance(corridor_name, str):
        return None
    name = corridor_name.strip().upper()
    # Expected "I-66 EB" or "I-66 WB"
    if " EB" in name or name.endswith("EB"):
        return "EB"
    if " WB" in name or name.endswith("WB"):
        return "WB"
    return None


def _month_from_filename(path: Path) -> Optional[str]:
    """
    Extract YYYY_MM from a filename like i66_toll_trip_pricing_2021_01.csv
    """
    m = re.search(r"(\d{4})_(\d{2})", path.name)
    if not m:
        return None
    return f"{m.group(1)}_{m.group(2)}"


def _safe_to_datetime(s: pd.Series) -> pd.Series:
    # Handles strings like 2021-10-01T09:24:00Z
    return pd.to_datetime(s, errors="coerce", utc=True)


def _build_probe_grid_for_day(day_utc: pd.Timestamp) -> pd.DatetimeIndex:
    """
    Build 5-minute timestamps for a UTC day: [00:00, 23:55] at 5min.
    """
    start = day_utc.normalize()
    end = start + pd.Timedelta(days=1) - pd.Timedelta(minutes=5)
    return pd.date_range(start=start, end=end, freq=PROBE_FREQ, tz="UTC")


def _build_probe_grid(min_ts: pd.Timestamp, max_ts: pd.Timestamp) -> pd.DatetimeIndex:
    """
    Build a continuous 5-minute grid from min_ts day start to max_ts day end.
    """
    if pd.isna(min_ts) or pd.isna(max_ts):
        return pd.DatetimeIndex([], tz="UTC")

    start_day = min_ts.normalize()
    end_day = max_ts.normalize()
    days = pd.date_range(start=start_day, end=end_day, freq="D", tz="UTC")

    grids = []
    for d in days:
        grids.append(_build_probe_grid_for_day(d))
    return grids[0].append(grids[1:]) if len(grids) > 1 else grids[0]


def _compute_dynamic_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    df must be sorted by ts_5min within direction.
    """
    df = df.copy()

    df["toll_change_5min"] = (
        df.groupby("direction")["toll_corridor_usd"]
        .diff(1)
        .astype("float64")
    )

    # 15 minutes = 3 intervals, 30 minutes = 6 intervals for 5-min data
    df["toll_rollmean_15min"] = (
        df.groupby("direction")["toll_corridor_usd"]
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .astype("float64")
    )

    df["toll_rollstd_30min"] = (
        df.groupby("direction")["toll_corridor_usd"]
        .rolling(window=6, min_periods=2)
        .std(ddof=0)
        .reset_index(level=0, drop=True)
        .astype("float64")
    )

    return df


def _build_silver_for_month(csv_path: Path) -> pd.DataFrame:
    print(f"Reading: {csv_path}")
    df = pd.read_csv(csv_path)

    required_cols = [
        "IntervalDateTime",
        "IntervalEndDateTime",
        "CorridorID",
        "CorridorName",
        "StartZoneID",
        "EndZoneID",
        "ZoneTollRate",
        "source_file",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {csv_path.name}: {missing}")

    # ------------------------------------------------------------
    # 1) Parse effective interval timestamps (UTC). This is truth.
    # ------------------------------------------------------------
    df["interval_start_utc"] = pd.to_datetime(df["IntervalDateTime"], utc=True, errors="coerce")
    df["interval_end_utc"] = pd.to_datetime(df["IntervalEndDateTime"], utc=True, errors="coerce")
    df = df.dropna(subset=["interval_start_utc"]).copy()

    # Snap interval start to the probe 5-min grid (handles :24, :36, etc.)
    df["toll_ts"] = df["interval_start_utc"].dt.floor(PROBE_FREQ)

    # ------------------------------------------------------------
    # 2) Direction and headline OD filter
    # ------------------------------------------------------------
    df["direction"] = df["CorridorName"].apply(_infer_direction)
    df = df[df["direction"].isin(["EB", "WB"])].copy()

    keep_mask = False
    for d, (sz, ez) in HEADLINE_PAIRS.items():
        keep_mask = keep_mask | (
            (df["direction"] == d)
            & (df["StartZoneID"].astype("int64") == int(sz))
            & (df["EndZoneID"].astype("int64") == int(ez))
        )
    df = df[keep_mask].copy()

    # Toll numeric
    df["toll_corridor_usd"] = pd.to_numeric(df["ZoneTollRate"], errors="coerce")
    df = df.dropna(subset=["toll_corridor_usd"]).copy()

    # Local date for grouping (DST-safe)
    df["toll_ts_local"] = df["toll_ts"].dt.tz_convert("America/New_York")
    df["local_date"] = df["toll_ts_local"].dt.date

    # If multiple records land in the same 5-min bin, keep max toll for safety
    df = (
        df.groupby(["direction", "toll_ts", "local_date"], as_index=False)
        .agg(
            corridor_id=("CorridorID", "first"),
            corridor_name=("CorridorName", "first"),
            start_zone_id=("StartZoneID", "first"),
            end_zone_id=("EndZoneID", "first"),
            toll_corridor_usd=("toll_corridor_usd", "max"),
            source_tag=("source_file", "first"),
        )
        .sort_values(["direction", "toll_ts"])
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------
    # 3) Expand to a 5-min grid per day starting at 05:30 local
    # ------------------------------------------------------------
    out_parts: List[pd.DataFrame] = []

    for direction, dft in df.groupby("direction"):
        for local_date, day_tbl in dft.groupby("local_date"):
            day_tbl = day_tbl.sort_values("toll_ts")

            start_local = pd.Timestamp(f"{local_date} 05:25", tz="America/New_York")
            start_utc = start_local.tz_convert("UTC")
            end_utc = day_tbl["toll_ts"].max()

            if end_utc < start_utc:
                continue

            grid = pd.date_range(start=start_utc, end=end_utc, freq=PROBE_FREQ, tz="UTC")
            g = pd.DataFrame({"ts_5min": grid})

            merged = pd.merge_asof(
                g.sort_values("ts_5min"),
                day_tbl.sort_values("toll_ts"),
                left_on="ts_5min",
                right_on="toll_ts",
                direction="backward",
                allow_exact_matches=True,
            ).drop(columns=["toll_ts"], errors="ignore")

            merged["direction"] = direction
            out_parts.append(merged)

    if not out_parts:
        return pd.DataFrame()

    silver = pd.concat(out_parts, ignore_index=True)
    silver = silver.sort_values(["direction", "ts_5min"]).reset_index(drop=True)

    # Local time for QA and time-of-day logic
    silver["ts_5min_local"] = silver["ts_5min"].dt.tz_convert("America/New_York")

    # ------------------------------------------------------------
    # 4) Directional toll active windows (fixes EB PM issue)
    # Adjust these if your official hours differ.
    # ------------------------------------------------------------
    AM_START = pd.to_datetime("05:30").time()
    AM_END = pd.to_datetime("09:30").time()
    PM_START = pd.to_datetime("15:30").time()
    PM_END = pd.to_datetime("19:00").time()

    t = silver["ts_5min_local"].dt.time
    silver["toll_active"] = 0

    # EB active in AM peak
    silver.loc[
        (silver["direction"] == "EB") & (t >= AM_START) & (t < AM_END),
        "toll_active"
    ] = 1

    # WB active in PM peak
    silver.loc[
        (silver["direction"] == "WB") & (t >= PM_START) & (t < PM_END),
        "toll_active"
    ] = 1

    # If inactive, null out toll value so it cannot leak into the model
    silver.loc[silver["toll_active"] == 0, "toll_corridor_usd"] = np.nan

    # Dynamic features (computed on toll series as stored)
    silver = _compute_dynamic_features(silver)

    cols = [
        "ts_5min",
        "ts_5min_local",
        "direction",
        "corridor_id",
        "corridor_name",
        "start_zone_id",
        "end_zone_id",
        "toll_corridor_usd",
        "toll_active",
        "toll_change_5min",
        "toll_rollmean_15min",
        "toll_rollstd_30min",
        "source_tag",
    ]
    for c in cols:
        if c not in silver.columns:
            silver[c] = np.nan
    silver = silver[cols]

    return silver



def main() -> None:
    # Collect monthly files
    csv_files = sorted(TOLL_MONTHLY_DIR.glob("i66_toll_trip_pricing_*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No matching CSV files found in: {TOLL_MONTHLY_DIR}")

    all_months = []

    for f in csv_files:
        ym = _month_from_filename(f) or "unknown_month"
        silver = _build_silver_for_month(f)

        if silver.empty:
            print(f"Skipping empty silver for {f.name}")
            continue

        out_path = OUT_DIR / f"toll_silver_{ym}.parquet"
        silver.to_parquet(out_path, index=False)
        print(f"Wrote: {out_path}  rows={len(silver):,}")

        all_months.append(silver.assign(year_month=ym))

    # Optional: write full-year file
    if all_months:
        full = pd.concat(all_months, ignore_index=True)

        # infer year from the first file tag, falls back to "year"
        year = "year"
        if "year_month" in full.columns and full["year_month"].notna().any():
            year = str(full["year_month"].dropna().iloc[0]).split("_")[0]

        out_year_parquet = OUT_DIR / f"toll_silver_{year}_all.parquet"
        out_year_csv = OUT_DIR / f"toll_silver_{year}_all.csv"

        full.to_parquet(out_year_parquet, index=False)
        full.to_csv(out_year_csv, index=False)

        print(f"Wrote: {out_year_parquet}  rows={len(full):,}")
        print(f"Wrote: {out_year_csv}  rows={len(full):,}")

In [3]:
if __name__ == "__main__":
    main()

Reading: C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data\a_raw_download\i66_toll_trip_pricing_monthly\csv_monthly\2023\i66_toll_trip_pricing_2023_01.csv
Wrote: C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data\b_processed_data\tolls\i66_itb\toll_silver_2023_01.parquet  rows=6,360
Reading: C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data\a_raw_download\i66_toll_trip_pricing_monthly\csv_monthly\2023\i66_toll_trip_pricing_2023_02.csv
Wrote: C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data\b_processed_data\tolls\i66_itb\toll_silver_2023_02.parquet  rows=6,042
Reading: C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data\a_raw_download\i66_toll_trip_pricing_monthly\csv_monthly\2023\i66_toll_trip_pricing_2023_03.csv
Wrote: C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data\b_processed_data\tolls\i66_itb\toll_silver_2023_03.parquet  rows=7,404
Reading